In [ ]:

import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

df = pd.read_csv("12thaugCustomer_Sentiment.csv")

print("Dataset Shape:", df.shape)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def clean_review(text):

    text = str(text).lower()

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    text = re.sub(
        r"\b[\w.-]+@[\w.-]+\.\w+\b",
        " ",
        text
    )

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\d+", " ", text)

    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    stemmed = [
        stemmer.stem(word)
        for word in tokens
    ]

    lemmatized = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return tokens, stemmed, lemmatized


df["tokens"] = df["review_text"].apply(
    lambda x: clean_review(x)[0]
)

df["stemmed_tokens"] = df["review_text"].apply(
    lambda x: clean_review(x)[1]
)

df["lemmatized_tokens"] = df["review_text"].apply(
    lambda x: clean_review(x)[2]
)

df["cleaned_review"] = df["lemmatized_tokens"].apply(
    lambda x: " ".join(x)
)


row = df.iloc[0]

print("\nExample Review")
print("Original      :", row["review_text"])
print("Tokens        :", row["tokens"])
print("Stemming      :", row["stemmed_tokens"])
print("Lemmatization :", row["lemmatized_tokens"])
print("Cleaned Review:", row["cleaned_review"])


df.to_csv(
    "final_cleaned_customer_reviews.csv",
    index=False
)

print("\nFinal Dataset Shape:", df.shape)
print("Saved successfully!")
